# Exercise 4B - Monitoring a Shifted Image Stream

**Goal:** connect pre-deployment robustness testing to deployment-time
monitoring.

In this notebook, a sequence of CIFAR-10 test windows simulates an
image stream. Blur severity increases gradually over time.

We monitor:

- raw-image brightness,
- raw-image sharpness,
- model confidence,
- predictive entropy,
- predicted accuracy, available here only for teaching,
- Kolmogorov-Smirnov drift statistics,
- a persistent alert rule.

A monitoring alert is an **investigation trigger**, not a diagnosis.

In [ ]:
from pathlib import Path
import os
import sys

os.environ.setdefault(
    "MPLCONFIGDIR",
    str((Path.cwd() / ".matplotlib_cache").resolve()),
)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageFilter

import torch
from torchvision import datasets

repo_root = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd().parent.parent
    if Path.cwd().name == "solutions"
    else Path.cwd()
)
sys.path.append(str(repo_root / "src"))

from cv4is_robustness.data import make_cifar10_bundle
from cv4is_robustness.models import (
    TinyCIFARNet,
    fit_model,
    get_device,
    load_checkpoint,
    save_checkpoint,
    set_seed,
)
from cv4is_robustness.evaluation import evaluate_corrupted_subset
from cv4is_robustness.monitoring import ks_statistic
from cv4is_robustness.visualization import plot_monitoring_table

SEED = 42
set_seed(SEED)
device = get_device()

data_root = repo_root / "data"
checkpoint_path = (
    repo_root / "artifacts" / "tiny_cifar10_checkpoint.pt"
)

print("Repository root:", repo_root)
print("Device:", device)

## Part A - Load the dataset and trained model

This notebook reuses the CIFAR-10 dataset and model checkpoint created in
Notebook 01.

Before continuing, run:

1. **Notebook 01, Part A** to download, verify, and extract CIFAR-10.
2. **Notebook 01, Part B** to train and save the model checkpoint.

In [ ]:
# Notebook 2 reuses the dataset and trained model from Notebook 1.

try:
    datasets.CIFAR10(
        root=data_root,
        train=True,
        download=False,
    )
    datasets.CIFAR10(
        root=data_root,
        train=False,
        download=False,
    )
except RuntimeError as error:
    raise FileNotFoundError(
        "CIFAR-10 was not found or failed its integrity check.\n\n"
        "Please open Notebook 01 and run Part A first to download, "
        "verify, and extract the dataset."
    ) from error


if not checkpoint_path.is_file():
    raise FileNotFoundError(
        f"Model checkpoint not found at:\n{checkpoint_path}\n\n"
        "Please open Notebook 01 and run Part B first to train and "
        "save the model checkpoint."
    )


bundle = make_cifar10_bundle(
    data_root=data_root,
    train_samples=10_000,
    val_samples=1_500,
    test_samples=2_000,
    batch_size=128,
    seed=SEED,
    num_workers=0,
)

model = TinyCIFARNet(
    num_classes=len(bundle.class_names)
).to(device)

load_checkpoint(
    model,
    checkpoint_path,
    device,
)

model.eval()

print("Loaded CIFAR-10 prepared in Notebook 01 Part A.")
print("Loaded model checkpoint created in Notebook 01 Part B.")

## Part B - Simulate a gradual shift

A monitoring window contains a batch of recent images:

$W_t = \{x_{t-n+1}, \ldots, x_t\}$

We use 12 non-overlapping windows. The first three are clean reference
windows. Blur then increases gradually.

Labels are kept only to reveal the true performance consequence during
this teaching exercise. In a real deployment, labels may be delayed or
unavailable.

In [ ]:
BLUR_RADIUS = [0.0, 0.4, 0.8, 1.2, 1.8, 2.5]


def stream_blur(
    image: Image.Image,
    severity: int,
    rng: np.random.Generator,
) -> Image.Image:
    del rng
    return image.filter(
        ImageFilter.GaussianBlur(
            radius=BLUR_RADIUS[severity]
        )
    )


WINDOW_SIZE = 120
WINDOW_SEVERITIES = [0, 0, 0, 1, 1, 2, 2, 3, 3, 4, 4, 5]

required_images = WINDOW_SIZE * len(WINDOW_SEVERITIES)
if required_images > len(bundle.test_indices):
    raise ValueError("Not enough fixed test indices for the stream.")

stream_windows = []

for window_id, severity in enumerate(WINDOW_SEVERITIES):
    start = window_id * WINDOW_SIZE
    stop = start + WINDOW_SIZE

    stream_windows.append(
        {
            "window": window_id,
            "severity": severity,
            "indices": bundle.test_indices[start:stop],
        }
    )

print("Number of windows:", len(stream_windows))
print("Images per window:", WINDOW_SIZE)

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(12, 9))
axes = axes.reshape(-1)

for ax, window in zip(axes, stream_windows):
    dataset_index = window["indices"][0]
    image, label = bundle.raw_test_dataset[dataset_index]
    corrupted = stream_blur(
        image,
        window["severity"],
        np.random.default_rng(SEED + window["window"]),
    )

    ax.imshow(corrupted)
    ax.set_title(
        f"window {window['window']}\n"
        f"severity {window['severity']}"
    )
    ax.axis("off")

plt.tight_layout()
plt.show()

## Part C - Raw image monitoring signals

### Task C1 - Mean brightness 🟢

Convert to grayscale and return the mean intensity in `[0, 1]`.

### Task C2 - Laplacian sharpness 🟡

Compute a simple discrete Laplacian and return its variance. A lower
value usually indicates a smoother or more blurred image.

Raw signals are interpretable, but each signal detects only certain
changes. Brightness may not respond strongly to blur.

In [ ]:
def mean_brightness(image: Image.Image) -> float:
    gray = np.asarray(
        image.convert("L"),
        dtype=np.float32,
    ) / 255.0
    return float(gray.mean())


def laplacian_sharpness(image: Image.Image) -> float:
    gray = np.asarray(
        image.convert("L"),
        dtype=np.float32,
    ) / 255.0

    center = gray[1:-1, 1:-1]
    laplacian = (
        -4.0 * center
        + gray[:-2, 1:-1]
        + gray[2:, 1:-1]
        + gray[1:-1, :-2]
        + gray[1:-1, 2:]
    )

    return float(laplacian.var())


test_image, _ = bundle.raw_test_dataset[
    bundle.test_indices[0]
]
print("Brightness:", mean_brightness(test_image))
print("Sharpness:", laplacian_sharpness(test_image))

In [ ]:
raw_rows = []
raw_values_by_window = {}

for window in stream_windows:
    brightness_values = []
    sharpness_values = []

    for dataset_index in window["indices"]:
        image, _ = bundle.raw_test_dataset[dataset_index]
        corrupted = stream_blur(
            image,
            window["severity"],
            np.random.default_rng(
                SEED + 1009 * dataset_index
            ),
        )

        brightness_values.append(
            mean_brightness(corrupted)
        )
        sharpness_values.append(
            laplacian_sharpness(corrupted)
        )

    raw_values_by_window[window["window"]] = {
        "brightness": np.asarray(brightness_values),
        "sharpness": np.asarray(sharpness_values),
    }

    raw_rows.append(
        {
            "window": window["window"],
            "severity": window["severity"],
            "brightness_mean": np.mean(
                brightness_values
            ),
            "sharpness_mean": np.mean(
                sharpness_values
            ),
        }
    )

raw_table = pd.DataFrame(raw_rows)
display(raw_table.round(4))

plot_monitoring_table(
    raw_table,
    ["brightness_mean", "sharpness_mean"],
    title="Raw image monitoring signals",
)

## Part D - Prediction monitoring signals

We now compute:

- accuracy, using labels only for retrospective teaching analysis,
- mean confidence,
- mean predictive entropy.

In live operation, confidence and entropy are available
immediately, while true performance may not be.

In [ ]:
prediction_rows = []

for window in stream_windows:
    metrics = evaluate_corrupted_subset(
        model=model,
        raw_dataset=bundle.raw_test_dataset,
        indices=window["indices"],
        corruption_fn=stream_blur,
        severity=window["severity"],
        eval_transform=bundle.eval_transform,
        device=device,
        batch_size=128,
        seed=SEED,
    )

    prediction_rows.append(
        {
            "window": window["window"],
            "severity": window["severity"],
            "accuracy": metrics["accuracy"],
            "mean_confidence": metrics["mean_confidence"],
            "mean_entropy": metrics["mean_entropy"],
        }
    )

prediction_table = pd.DataFrame(prediction_rows)
display(prediction_table.round(3))

plot_monitoring_table(
    prediction_table,
    ["accuracy", "mean_confidence", "mean_entropy"],
    title="Prediction and retrospective performance signals",
)

## Part E - Compare each window with the clean reference

The Kolmogorov-Smirnov statistic measures the largest difference
between two empirical cumulative distributions.

We pool the first three clean windows into the reference
distribution.

### Task E1 - Compute window-level KS statistics 🟡

For each window, compare its brightness and sharpness values with
the corresponding reference distribution.

In [ ]:
reference_window_ids = [0, 1, 2]

reference_brightness = np.concatenate(
    [
        raw_values_by_window[window_id]["brightness"]
        for window_id in reference_window_ids
    ]
)
reference_sharpness = np.concatenate(
    [
        raw_values_by_window[window_id]["sharpness"]
        for window_id in reference_window_ids
    ]
)


def compute_window_ks_table(
    raw_values_by_window: dict,
) -> pd.DataFrame:
    rows = []

    for window_id, values in raw_values_by_window.items():
        rows.append(
            {
                "window": window_id,
                "brightness_ks": ks_statistic(
                    reference_brightness,
                    values["brightness"],
                ),
                "sharpness_ks": ks_statistic(
                    reference_sharpness,
                    values["sharpness"],
                ),
            }
        )

    return pd.DataFrame(rows).sort_values("window")


ks_table = compute_window_ks_table(
    raw_values_by_window
)
display(ks_table.round(3))

In [ ]:
monitoring_table = (
    raw_table
    .merge(prediction_table, on=["window", "severity"])
    .merge(ks_table, on="window")
)

display(monitoring_table.round(3))

plot_monitoring_table(
    monitoring_table,
    ["brightness_ks", "sharpness_ks"],
    title="KS drift statistics against the clean reference",
)

## Part F - Persistent drift alarms

Lecture 5 uses the rule:


$alert \quad \text{if} \quad d_t > \tau \quad \text{for } r \text{ consecutive windows}$

Persistence reduces one-window false alarms.

### Task F1 - Implement the alert state 🟢

The output should become `True` once the threshold has been
exceeded for `persistence` consecutive windows. Reset the counter
when the statistic falls below the threshold.

In [ ]:
def persistent_alerts(
    values,
    threshold: float,
    persistence: int,
) -> list[bool]:
    alerts = []
    consecutive = 0

    for value in values:
        if value > threshold:
            consecutive += 1
        else:
            consecutive = 0

        alerts.append(consecutive >= persistence)

    return alerts


SHARPNESS_KS_THRESHOLD = 0.30
PERSISTENCE = 2

monitoring_table["sharpness_alert"] = persistent_alerts(
    monitoring_table["sharpness_ks"].to_numpy(),
    threshold=SHARPNESS_KS_THRESHOLD,
    persistence=PERSISTENCE,
)

display(
    monitoring_table[
        [
            "window",
            "severity",
            "sharpness_ks",
            "sharpness_alert",
            "accuracy",
            "mean_confidence",
            "mean_entropy",
        ]
    ].round(3)
)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.5))

ax.plot(
    monitoring_table["window"],
    monitoring_table["sharpness_ks"],
    marker="o",
    label="sharpness KS",
)
ax.axhline(
    SHARPNESS_KS_THRESHOLD,
    linestyle="--",
    label="alert threshold",
)

alert_windows = monitoring_table.loc[
    monitoring_table["sharpness_alert"],
    "window",
]
for window_id in alert_windows:
    ax.axvline(window_id, alpha=0.2)

ax.set_xlabel("Window")
ax.set_ylabel("KS statistic")
ax.set_title("Persistent sharpness-drift alert")
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

## Final interpretation

Answer in 8-10 sentences:

1. Which raw signal detects the simulated blur?
2. Why does brightness perform differently from sharpness?
3. At which window does the persistent alert begin?
4. Does the first alert coincide exactly with a large accuracy drop?
5. Does confidence reveal the full performance loss?
6. What information is unavailable without labels?
7. Why should the alert trigger investigation rather than immediate
   retraining?
8. What additional signals would you monitor in an industrial
   image pipeline?

In [ ]:
answer = """
Sharpness is the clearest raw-image signal for the simulated blur because
the Laplacian variance decreases as image edges are smoothed. Mean brightness
changes very little because blur redistributes local intensity without strongly
changing the overall image brightness. With the chosen threshold and persistence
rule, the alert begins at window 6. The large accuracy drop starts one window
earlier, so the alert does not coincide exactly with the first performance loss.
Confidence does not reveal the full degradation and even increases again under
strong blur while accuracy remains very low. Without labels, true accuracy,
class-specific recall, and the operational cost of errors cannot be measured
directly. The alert should trigger investigation rather than immediate retraining
because the cause may be a focus problem, dirty lens, temporary disturbance, or
sensor issue. An industrial monitoring pipeline should combine raw image
statistics, preprocessing checks, model outputs, uncertainty, embeddings,
sensor metadata, review rates, and delayed labeled performance.
"""
print(answer)

## Optional extensions 🔴

- Replace gradual blur with an abrupt camera-change simulation.
- Simulate recurring seasonal lighting.
- Compare KS thresholds chosen from clean-window quantiles.
- Add a predicted-class-distribution PSI signal.
- Monitor a feature embedding instead of only raw image statistics.
- Introduce delayed labels and compute a retrospective performance
  dashboard.
- Create a compound incident: brightness shift followed by blur.